<a href="https://colab.research.google.com/github/RabbicseJnu/Data_Science_Project/blob/main/SARIMA_%26_LSTM_For_LPG_Usage_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas numpy matplotlib statsmodels scikit-learn

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("/lpg_dataset_2021_2025_hourly.csv")

# Convert datetime
df['Datetime'] = pd.to_datetime(df['Datetime'], dayfirst=True)

# Sort & set index
df = df.sort_values('Datetime')
df.set_index('Datetime', inplace=True)

# Optional: filter one household
df = df[df['Household_ID'] == 'HH_1']

# Ensure hourly frequency
df = df.asfreq('H')

# Fill missing values
df['Hourly_Usage_kg'].fillna(method='ffill', inplace=True)

print(df.head())

                    Household_ID  Gas_Level_kg  Hourly_Usage_kg  Refill_Event  \
Datetime                                                                        
2021-01-01 00:00:00         HH_1         11.44              0.0             0   
2021-01-01 01:00:00         HH_1         11.44              0.0             0   
2021-01-01 02:00:00         HH_1         11.44              0.0             0   
2021-01-01 03:00:00         HH_1         11.44              0.0             0   
2021-01-01 04:00:00         HH_1         11.44              0.0             0   

                     Hour  Month  
Datetime                          
2021-01-01 00:00:00     0      1  
2021-01-01 01:00:00     1      1  
2021-01-01 02:00:00     2      1  
2021-01-01 03:00:00     3      1  
2021-01-01 04:00:00     4      1  


/tmp/ipykernel_50614/4033733963.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.asfreq('H')
/tmp/ipykernel_50614/4033733963.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Hourly_Usage_kg'].fillna(method='ffill', inplace=True)
/tmp/ipykernel_50614/4033733963.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Hourly_Usage_kg'].fillna(method='ffill', inplace=Tr

In [ ]:
train = df[:-168]   # all except last 7 days
test = df[-168:]    # last 7 days

# SARIMA Baseline
!pip install pmdarima
import pmdarima as pm

# Fit auto_arima to find optimal SARIMA parameters
sarima_model = pm.auto_arima(
    train['Hourly_Usage_kg'],
    seasonal=True,
    m=24,  # daily pattern
    d=None,  # let model decide differencing
    D=None,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True,
    max_p=3, max_q=3,
    max_P=2, max_Q=2
)

print(sarima_model.summary())

Performing stepwise search to minimize aic
 ARIMA(2,1,2)(1,0,1)[24] intercept   : AIC=-346996.502, Time=562.72 sec
 ARIMA(0,1,0)(0,0,0)[24] intercept   : AIC=-291599.265, Time=8.23 sec
 ARIMA(1,1,0)(1,0,0)[24] intercept   : AIC=inf, Time=160.94 sec


In [ ]:
forecast = sarima_model.predict(n_periods=168)

# Convert to DataFrame
forecast_index = pd.date_range(start=test.index[0], periods=168, freq='H')

forecast_df = pd.DataFrame({
    'Datetime': forecast_index,
    'Predicted_Usage': forecast
})

forecast_df.set_index('Datetime', inplace=True)

print(forecast_df.head())

NameError: name 'sarima_model' is not defined